# Lab 09: Reactive Agentic Analytics Dashboard (Streamlit + CrewAI)

**Course / Disciplina:** Augmented Analytics & AI-Driven Insights (FIAP MBA)  
**[🇧🇷 PT-BR]** Criar uma interface reativa (Dashboard) para a Squad de Agentes utilizando Streamlit e PyNgrok.  
**[🇺🇸 EN]** Build a reactive executive cockpit (Dashboard) for the Multi-Agent Squad using Streamlit and PyNgrok.

---
### 🚀 1. Dependency Installation & Sample Ingestion / Instalação de Dependências

In [ ]:
!pip install crewai streamlit pyngrok pandas -q
import urllib.request, os
if not os.path.exists('PesquisaClientes.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/rafaelmatsuyama/FIAP-BI-AugmentedAnalytics/main/lab01-cx-analytics/Lab%2001%20-%20PesquisaClientes.csv', 'PesquisaClientes.csv')
    print('✅ Demo dataset downloaded successfully! / Dataset de demonstração baixado com sucesso!')

### 📝 2. Dashboard Application Synthesis (`app.py`) / Desenvolvimento do Dashboard
* 🇧🇷 A célula abaixo grava o arquivo `app.py` no ambiente do Colab com persistência em `st.session_state` e carregamento automático do dataset padrão.
* 🇺🇸 The cell below writes `app.py` to the Colab workspace with `st.session_state` persistence and automatic fallback to the default demo dataset.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os
from crewai import Agent, Task, Crew, Process

# -----------------------------------------------------------------
# 1. UI CONFIGURATION
# -----------------------------------------------------------------
st.set_page_config(page_title="BI Agentic Dashboard", layout="wide")

st.title("🤖 Agentic Analytics Dashboard")
st.subheader("Transforming Telemetry into Executive Decisions with AI Squads")

with st.sidebar:
    st.header("⚙️ Control Panel / Configurações")
    api_key = st.text_input("🔑 Gemini API Key:", type="password")
    model_choice = st.selectbox("🧠 Model / Modelo:", ["gemini/gemini-3.8-flash"])
    output_lang = st.selectbox("🌐 Output Language / Idioma:", ["Portuguese (Brazil)", "English"])
    st.divider()
    st.info("MBA in BI & Analytics - FIAP")

# -----------------------------------------------------------------
# 2. DATA INGESTION (UPLOAD OR SELF-HEALING DEFAULT)
# -----------------------------------------------------------------
uploaded_file = st.file_uploader("📂 Upload CSV dataset (or leave empty to use default PesquisaClientes.csv):", type="csv")
data_source = uploaded_file if uploaded_file is not None else ('PesquisaClientes.csv' if os.path.exists('PesquisaClientes.csv') else None)

if data_source and api_key:
    df = pd.read_csv(data_source)
    if uploaded_file is None:
        st.caption("ℹ️ Using pre-loaded Colab dataset: `PesquisaClientes.csv` / Usando base padrão do Colab.")
    
    # Executive KPI Cards
    col1, col2, col3 = st.columns(3)
    col1.metric("📊 Total Records / Registros", len(df))
    col2.metric("📋 Total Columns / Colunas", len(df.columns))
    col3.metric("🔢 Numeric Columns / Numéricas", len(df.select_dtypes(include='number').columns))
    
    st.write("### 🔍 Telemetry Sample / Amostra dos Dados Carregados", df.head(5))
    
    # Trigger with session_state persistence
    if st.button("🚀 Launch Agent Squad / Disparar Squad de Agentes"):
        os.environ["GEMINI_API_KEY"] = api_key
        os.environ["GOOGLE_API_KEY"] = api_key
        
        with st.spinner("🤖 Agents analyzing telemetry... / Agentes analisando dados... Aguarde."):
            analyst = Agent(
                role='Senior Strategic Data Analyst',
                goal=f'Identify critical business patterns in the provided telemetry (respond in {output_lang})', 
                backstory='Expert in statistical modeling, customer behavior, and executive boardroom synthesis.',
                llm=model_choice
            )
            
            task = Task(
                description=f'Analyze the variables {df.columns.tolist()} in the following sample and synthesize an executive brief in {output_lang}: {df.head(10).to_markdown()}',
                expected_output=f'A Markdown executive report in {output_lang} containing 3 actionable strategic insights.',
                agent=analyst
            )
            
            crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
            raw_result = crew.kickoff()
            st.session_state["relatorio_analise"] = str(raw_result)

    # Persistent display and Markdown export
    if "relatorio_analise" in st.session_state:
        st.success("✅ Analysis Completed! / Análise Concluída!")
        st.markdown("--- ")
        st.markdown(st.session_state["relatorio_analise"])
        
        st.download_button(
            label="📥 Download Executive Dossier (.md) / Baixar Dossiê Executivo",
            data=st.session_state["relatorio_analise"],
            file_name="dossie_executivo_bi.md",
            mime="text/markdown"
        )

elif not api_key:
    st.warning("⚠️ Please enter your Gemini API Key in the sidebar to begin. / Insira sua API Key no menu lateral para começar.")

### 🌐 3. Execution & Secure Tunneling (PyNgrok) / Execução & Túnel
* 🇧🇷 Insira seu Token do Ngrok no campo abaixo e rode a célula para abrir o link público do Dashboard.
* 🇺🇸 Enter your Ngrok AuthToken below and run the cell to expose the live Dashboard via a secure HTTPS tunnel.

In [ ]:
# @title Ngrok Tunnel Configuration / Configuração do Túnel
NGROK_TOKEN = "" # @param {type:"string"}

from pyngrok import ngrok
import os

if NGROK_TOKEN:
    # Terminate stale tunnels and open port 8501
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN)
    
    print("⌛ Establishing secure tunnel... / Estabelecendo conexão segura...")
    public_url = ngrok.connect(8501).public_url
    print(f"🔗 LIVE DASHBOARD URL / DASHBOARD ATIVO EM: {public_url}")
    print("--- ")
    
    # Launch Streamlit server
    !streamlit run app.py --server.port 8501
else:
    print("❌ ERROR: Please provide your NGROK_TOKEN above. / ERRO: Insira o NGROK_TOKEN para gerar o link.")